In [1]:
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
from PIL import Image
from torch.optim import Adam, SGD, lr_scheduler
import matplotlib.pyplot as plt
from torchvision import transforms, models
from UNET_LIB.Unet import UNet
from utils import DiceLoss, SquarePad, SquarePad255
from Liver_Dataset import Liver_Dataset
from torch.utils.data import Dataset, Subset, DataLoader

train_total=4000
mult_spec={
    'perfe':[1,2,4,8,16,32,50],
    'poly+':[1,2,4,8,16,32,train_total/94],
    'poly-':[1,2,4,8,train_total/250],
    'rough':[1,2,4,8,train_total/276],
    'bbox_msk':[1,2,4,8,10],
    'sam_box':[1,2,4,8,10],
    'point_msk':[1,2,4,train_total/562]
}
group_spec={
    'perfe':80,
    'poly+':94,
    'poly-':250,
    'rough':276,
    'bbox_msk':400,
    'sam_box':400,
    'point_msk':562
}

epochs0 = 40
pretrained = 'ub'
assert pretrained in ['pb','fb','ub']


img_preprocess = transforms.Compose([    
#     SquarePad(),
#     transforms.Resize(512),
    transforms.RandomEqualize(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(0,1),
])

msk_preprocess = transforms.Compose([
#     SquarePad(),
#     transforms.Resize(512, \
#             interpolation = transforms.InterpolationMode.NEAREST),
])

In [2]:

num_classes = 2

for label_type in ['poly-']:# ['perfe','poly+','poly-','rough','bbox_msk']:
    group_size = group_spec[label_type]
    DATA = Liver_Dataset('./Liver/','trainval',label_type, img_preprocess, msk_preprocess,crop=True,crop_size=352)

    for multiplicity in mult_spec[label_type][0:3]:
        
        
        model = UNet(in_channels = 1, num_classes=num_classes)
        loss_fn = nn.CrossEntropyLoss(ignore_index=255) #DiceLoss()


        model = nn.DataParallel(model).cuda()
        model_name = f'UNet_L{label_type}_m{round(multiplicity)}_{pretrained}' 

        train_subset = Subset(DATA, list(range(round((group_size*multiplicity)))))
        val_subset = Subset(DATA, list(range(train_total, len(DATA))))
        train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, drop_last=True,num_workers=32,pin_memory=True)
        val_loader = DataLoader(val_subset, batch_size=32, shuffle=False, drop_last=True,num_workers=16,pin_memory=True)

        epochs = round(epochs0*(1.5**np.log2(train_total/len(train_subset))))
        
        min_loss = np.inf
        fin_epoch = 0
        if pretrained == 'pb':
            optimizer = Adam(model.parameters(),lr=1e-4,weight_decay=1e-6)

        elif pretrained == 'fb':
            optimizer = SGD(model.module.classifier.parameters(),lr=1.5e-2,momentum=0.9,weight_decay=1e-3)
            scheduler = lr_scheduler.MultiStepLR(optimizer, milestones=[2000*i for i in range(1,5)], gamma=0.1)
        elif pretrained == 'ub':
            optimizer = SGD(model.parameters(),lr=1e-2,momentum=0.9,weight_decay=1e-5)
            scheduler = lr_scheduler.MultiStepLR(optimizer, milestones=[3000*i for i in range(1,4)], gamma=0.1)

        else:
            assert False

        try:
                checkpoint = torch.load(f'./model_checkpoints/{model_name}.pth')
                fin_epoch = checkpoint['fin_epoch']
                min_loss = checkpoint['min_loss']
                model.load_state_dict(checkpoint['model_state_dict'])
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
                print('optimizer loaded')
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
                print('scheduler loaded')
        except:
            if fin_epoch ==0:
                print(f'new model training {model_name}')
        pbar = tqdm(range(epochs-fin_epoch))
        for e in pbar:
            train_loss, val_loss = 0, 0


            model.train()
            for idx,(X,y) in enumerate(train_loader):

                with torch.no_grad():
                    valid_loc = y.cuda()
                    y = y.squeeze().cuda()
                optimizer.zero_grad()

                yhat = model(X.contiguous().cuda())
                loss = (loss_fn(yhat,y)*valid_loc).mean()
                loss.backward()
                optimizer.step()
                if pretrained in ['fb','ub']:
                    scheduler.step()

                train_loss += loss.item()
            train_loss /= (idx+1)

            model.eval()
            with torch.no_grad():
                class_intersect = np.zeros((num_classes,),dtype='float')
                class_union= np.zeros((num_classes,),dtype='float')
                for idx,(X,y) in enumerate(val_loader):
                    y = y.cuda().contiguous().flatten()

                    yhat = model(X.contiguous().cuda())
                    yhat_lab = torch.argmax(yhat, dim=1).flatten()
                    yhat_lab[y == 255] = 255

                    for j in range(num_classes):

                        y_bi = y == j
                        yhat_bi = yhat_lab == j
                        I = ((y_bi * yhat_bi).sum()).item()
                        U = (y_bi.sum() + yhat_bi.sum() - I).item()
                        assert I <= U
                        class_intersect[j] += I
                        class_union[j] += U

                IOUs = class_intersect/class_union
                val_loss=-np.mean(IOUs)
                pbar.set_description(f'Train loss: {train_loss}| IOU: {-val_loss}')

                #Dynamic class weight adjustment for loss function
                boost = 1/np.clip(IOUs,5e-2,1)
                boost = torch.tensor(boost/boost.sum(),dtype=torch.float).cuda()

            loss_fn = nn.CrossEntropyLoss(boost, ignore_index=255)   

            if e <10:
                continue

            if val_loss < min_loss:
                min_loss = val_loss
                to_save ={
                    'min_loss': min_loss,
                    'fin_epoch': e+fin_epoch+1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict()}
                if pretrained in ['fb','ub']:
                    to_save['scheduler_state_dict']= scheduler.state_dict()

                torch.save(to_save, f'./model_checkpoints/{model_name}.pth')  


optimizer loaded
scheduler loaded


Train loss: 0.0150517325848341| IOU: 0.7321900326123156: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 134/134 [34:14<00:00, 15.33s/it]


optimizer loaded
scheduler loaded


Train loss: 0.00836341590019724| IOU: 0.8160385413423763: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 67/67 [23:36<00:00, 21.13s/it]


optimizer loaded
scheduler loaded


Train loss: 0.00807022056992977| IOU: 0.8495787557840991: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [12:18<00:00, 32.09s/it]


In [2]:
pretrained='ub'
performance={}
num_classes=2
test_preprocess = transforms.Compose([    
#     SquarePad(),
#     transforms.Resize(512),
    transforms.ToTensor(),
    transforms.Normalize(0,1),
])

test_loader = DataLoader(Liver_Dataset('./Liver/','test','perfe', 
                                       test_preprocess, msk_preprocess,crop=True,crop_size=512), 
                         batch_size=8, shuffle=False, num_workers=8)
model = UNet(in_channels = 1, num_classes=num_classes)
model = nn.DataParallel(model).cuda()

for label_type in mult_spec.keys():
    performance[label_type]={}
    
    for multiplicity in mult_spec[label_type]:
        
        model_name = f'UNet_L{label_type}_m{round(multiplicity)}_{pretrained}' 
        checkpoint = torch.load(f'.//model_checkpoints/{model_name}.pth')
        model.load_state_dict(checkpoint['model_state_dict'])
        
        model.eval()
        print(f'Working on label_type={label_type}:{round(multiplicity)}')
        class_intersect = np.zeros((num_classes, ),dtype='float')
        class_union = np.zeros((num_classes, ),dtype='float')
    
        with torch.no_grad():
            for idx,(X,y) in enumerate(test_loader):
                y = y.cuda().contiguous().flatten()
                
                yhat = model(X.contiguous().cuda())
                yhat_lab = torch.argmax(yhat, dim=1).flatten()
                yhat_lab[y == 255] = 255

                for j in range(num_classes):

                    y_bi = y == j
                    yhat_bi = yhat_lab == j
                    I = ((y_bi * yhat_bi).sum()).item()
                    U = (y_bi.sum() + yhat_bi.sum() - I).item()
                    assert I <= U
                    class_intersect[j] += I
                    class_union[j] += U
            
        performance[label_type][multiplicity]=(class_intersect, class_union)
        
np.save(f'Liver_Inter-Union_{pretrained}2', performance)
print(performance)

Working on label_type=perfe:1
Working on label_type=perfe:2
Working on label_type=perfe:4
Working on label_type=perfe:8
Working on label_type=perfe:16
Working on label_type=perfe:32
Working on label_type=perfe:50
Working on label_type=poly+:1
Working on label_type=poly+:2
Working on label_type=poly+:4
Working on label_type=poly+:8
Working on label_type=poly+:16
Working on label_type=poly+:32
Working on label_type=poly+:43
Working on label_type=poly-:1
Working on label_type=poly-:2
Working on label_type=poly-:4
Working on label_type=poly-:8
Working on label_type=poly-:16
Working on label_type=rough:1
Working on label_type=rough:2
Working on label_type=rough:4
Working on label_type=rough:8
Working on label_type=rough:14
Working on label_type=bbox_msk:1
Working on label_type=bbox_msk:2
Working on label_type=bbox_msk:4
Working on label_type=bbox_msk:8
Working on label_type=bbox_msk:10
Working on label_type=sam_box:1
Working on label_type=sam_box:2
Working on label_type=sam_box:4
Working on

In [8]:
for i in [1,2,4,8,16]:
    a,b = performance['poly+'][i]
    print(a/b)

[0.96599602 0.54922103]
[0.97156139 0.572518  ]
[0.98327888 0.71140389]
[0.98696433 0.77316361]
[0.9893019  0.81179906]


In [5]:
for i in [1,2,4,7.117437722419929]:
    a,b = performance['point_msk'][i]
    print(a/b)

[0.96464071 0.51870206]
[0.96504992 0.54381073]
[0.96364097 0.56335606]
[0.96659081 0.57457342]


In [4]:
#snipping result
test_preprocess = transforms.Compose([    
#     SquarePad(),
#     transforms.Resize(512),
    transforms.ToTensor(),
    transforms.Normalize(0,1),
])
num_classes=2
label_type='perfe'
test_loader = DataLoader(Liver_Dataset('/data/Liver/','test','perfe', test_preprocess, msk_preprocess,crop=True,crop_size=512), batch_size=8, shuffle=False)

for multiplicity in mult_spec[label_type]:
        
    model = UNet(in_channels = 1, num_classes=num_classes)
    model = nn.DataParallel(model).cuda()
        
    model_name = f'UNet_L{label_type}_m{round(multiplicity)}_{pretrained}'  
    try:
        checkpoint = torch.load(f'/data/model_checkpoints/{model_name}.pth')
        model.load_state_dict(checkpoint['model_state_dict'])
        print('Loaded', model_name)
    except:
        print(f'Error loading {model_name}')
        assert False

    model.eval()
    with torch.no_grad():    
        class_intersect = np.zeros((num_classes,),dtype='float')
        class_union= np.zeros((num_classes,),dtype='float')

        for idx,(X,y) in enumerate(test_loader):
            y = y.flatten()

            yhat = model(X.contiguous().cuda())
            yhat_lab = torch.argmax(yhat.cpu(), dim=1).flatten()
            skip_id = np.argwhere(y == 255)
            yhat_lab[skip_id] = 255

            for j in range(num_classes):

                y_bi = y == j
                yhat_bi = yhat_lab == j
                I = (y_bi * yhat_bi).sum()
                U = y_bi.sum() + yhat_bi.sum() - I
                assert I <= U
                class_intersect[j] += I
                class_union[j] += U

        IOUs = class_intersect/class_union
        val_loss=-np.mean(IOUs)
        print('IOU', -val_loss)
            

Loaded UNet_Lperfe_m1_ub


/opt/conda/lib/python3.8/site-packages/torch/nn/functional.py:3454: UserWarning: Default upsampling behavior when mode=bilinear is changed to align_corners=False since 0.4.0. Please specify align_corners=True if the old behavior is desired. See the documentation of nn.Upsample for details.
  warnings.warn(


IOU 0.568952984809014
Loaded UNet_Lperfe_m2_ub
IOU 0.47490772583258145
Loaded UNet_Lperfe_m4_ub
IOU 0.7504934517838866
Loaded UNet_Lperfe_m8_ub
IOU 0.8195960732462702
Loaded UNet_Lperfe_m16_ub
IOU 0.8935292368245595
Loaded UNet_Lperfe_m32_ub
IOU 0.8978911555199378
Loaded UNet_Lperfe_m50_ub
IOU 0.9007755978616656


In [5]:
IOUs

array([0.98898049, 0.8125707 ])